In [ ]:
%load_ext autoreload 
%autoreload 2

In [ ]:
import numpy as np
import datetime
from stonesoup.types.array import StateVector, CovarianceMatrix
from stonesoup.types.state import State, GaussianState

from stonesoup.models.transition.linear import (
    CombinedLinearGaussianTransitionModel, ConstantVelocity, KnownTurnRate)
from stonesoup.models.transition.nonlinear import kinematic_bicycle

from stonesoup.simulator.simple import SwitchMultiTargetGroundTruthSimulator, MultiTargetGroundTruthSimulator,  SimpleDetectionSimulator
from stonesoup.types.state import GaussianState
from stonesoup.models.measurement.linear import LinearGaussian
from stonesoup.types.detection import Detection
from itertools import tee
from stonesoup.plotter import AnimatedPlotterly, Plotterly
from tqdm.notebook import trange, tqdm


In [ ]:
np.random.seed(42)
number_steps             = 30
start_time               = datetime.datetime.fromtimestamp(1767225600)
initial_state_mean       = StateVector([[0], [0], [0], [0]])
initial_state_covariance = CovarianceMatrix(np.diag([10, 5, 10, 5]))
timestep_size            = datetime.timedelta(seconds=1/15)
initial_state            = GaussianState(initial_state_mean, initial_state_covariance,timestamp=start_time)

#Define the generating motion model
constant_velocity        = CombinedLinearGaussianTransitionModel([ConstantVelocity(0.05), ConstantVelocity(0.05)])
turn_left                = KnownTurnRate([0.05, 0.05], np.radians(90))
turn_right               = KnownTurnRate([0.05, 0.05], np.radians(-90))
model_probs              = np.array([[0.9, 0.05, 0.05],  # keep straight, turn left, turn right
                                     [0.1, 0.9, 0.0],  # go straight, keep turning left, turn right
                                     [0.1, 0.0, 0.9]])  # go straight, turn left, keep turning right
n_truths =   3
xmin     = -40
xmax     =  40
ymin     = -40
ymax     =  40
preexisting_states = []

for i in range(0, n_truths):
    x = np.random.randint(xmin, xmax)   # x position of initial state
    y = np.random.randint(ymin, ymax)   # y position of initial state
    y_vel = np.random.randint(-100, 100) / 1.0  # x velocity will start between -2 and 2
    x_vel = np.random.randint(-100, 100) / 1.0  # y velocity will start between -2 and 2
    preexisting_states.append(StateVector([x, x_vel, y, y_vel]))


# Now we have initialised everything, so we can generate the ground truth:
ground_truth_gen = SwitchMultiTargetGroundTruthSimulator(
    initial_state=initial_state,
    transition_models=[constant_velocity, turn_left, turn_right],
    #transition_models=[constant_velocity, constant_velocity, constant_velocity],
    model_probs=model_probs,  # put in matrix from above
    number_steps=number_steps,  # how long we want each track to be
    timestep=timestep_size,
    birth_rate=0,
    death_probability=0.05,
    preexisting_states=preexisting_states,
    seed=42
)


In [ ]:
gen_measurement_model = LinearGaussian(ndim_state=4,mapping=(0, 2), noise_covar=np.array([[0.1, 0],[0, 0.1]]))
meas_range            = np.array([[xmin, xmax], [ymin , ymax]])
detector              = SimpleDetectionSimulator(ground_truth_gen,gen_measurement_model,meas_range=meas_range, detection_probability=1,clutter_rate=0,seed=42)

detections            = set()
ground_truth          = set()
detection_list        = []
timesteps             = []

for time, dets in tqdm(detector):
    detections   |= dets
    ground_truth |= ground_truth_gen.groundtruth_paths
    detection_list.append((time,dets))
    timesteps.append(time)


In [ ]:
from stonesoup.reader.base import DetectionReader
from stonesoup.base import Property
from stonesoup.buffered_generator import BufferedGenerator
from stonesoup.simulator import DetectionSimulator

class DetectionCache(DetectionReader):

    detection_list:list = Property()
    strip_measurement_model:bool = Property(default=True)

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.current = [None,None]

    @BufferedGenerator.generator_method
    def detections_gen(self):
        for i, (time, dets) in enumerate(self.detection_list):
            if(self.strip_measurement_model):
                for det in dets:
                    det.measurement_model=None
            self.current = [time,dets]
            yield time, dets

detector = DetectionCache(detection_list, strip_measurement_model=True)

In [ ]:
plotter = AnimatedPlotterly(timesteps=timesteps)
plotter.plot_ground_truths(ground_truth, [0, 2])
plotter.plot_measurements(detections, [0, 2],measurement_model=gen_measurement_model)
plotter.fig


In [ ]:
from stonesoup.predictor.particle import ParticlePredictor
from stonesoup.resampler.particle import ESSResampler
from stonesoup.updater.particle import ParticleUpdater
from stonesoup.hypothesiser.distance import DistanceHypothesiser
from stonesoup.measures import Mahalanobis
from stonesoup.dataassociator.neighbour import GNNWith2DAssignment
from stonesoup.deleter.time import UpdateTimeDeleter
from stonesoup.initiator.simple import GaussianParticleInitiator
from stonesoup.types.state import GaussianState
from stonesoup.initiator.simple import SimpleMeasurementInitiator
from stonesoup.tracker.simple import MultiTargetTracker
from stonesoup.models.transition.nonlinear import kinematic_bicycle2, CTRV

tracking_measurement_model= LinearGaussian(ndim_state=5,mapping=(0, 1), noise_covar=np.array([[4, 0], [0, 4]]))
transition_model_estimate = kinematic_bicycle2(std_accl=1, std_accd=1,L=1)
predictor_PF              = ParticlePredictor(transition_model_estimate)
resampler                 = ESSResampler()
updater_PF                = ParticleUpdater(measurement_model=tracking_measurement_model, resampler=resampler)
hypothesiser_PF           = DistanceHypothesiser(predictor_PF, updater_PF, measure=Mahalanobis(), missed_distance=5)
data_associator_PF        = GNNWith2DAssignment(hypothesiser_PF)
deleter                   = UpdateTimeDeleter(datetime.timedelta(seconds=1), delete_last_pred=True)
prior_state               = GaussianState(StateVector([0, 0, 0, 0, 0]), np.diag([10, 10, 10, 2*np.pi, np.pi/4]) ** 2)
initiator_Part            = SimpleMeasurementInitiator(prior_state, measurement_model=tracking_measurement_model,skip_non_reversible=True)
initiator_PF              = GaussianParticleInitiator(number_particles=10000,initiator=initiator_Part, use_fixed_covar=False)
tracker_PF                = MultiTargetTracker(initiator=initiator_PF, deleter=deleter,detector=detector,data_associator=data_associator_PF,updater=updater_PF)

tracks_PF = set()
timesteps = []
for time, current_tracks in tqdm(tracker_PF,total=number_steps):
    tracks_PF.update(current_tracks)
    timesteps.append(time)


In [ ]:
plotter = AnimatedPlotterly(timesteps=timesteps, tail_length=0.3)
#plotter.plot_measurements(detections, [0, 2], measurement_model=gen_measurement_model)
#plotter.plot_ground_truths(ground_truth, [0, 2])
plotter.plot_tracks(tracks_PF, [0, 1], label="PF", uncertainty=False, particle=True)
plotter.fig

In [ ]:
plotter = AnimatedPlotterly(timesteps, tail_length=1/15)
plotter.plot_ground_truths(ground_truth, [0, 2])
plotter.plot_track_headings(tracks_PF, [0, 1, 2, 3], plot_history=True, velocity_scale=1.0)
plotter.fig

In [ ]:
tracking_measurement_model= LinearGaussian(ndim_state=5,mapping=(0, 1), noise_covar=np.array([[2, 0], [0, 2]]))
transition_model_estimate = CTRV(linear_noise_coeff=1, turn_noise_coeff=1)
predictor_PF              = ParticlePredictor(transition_model_estimate)
resampler                 = ESSResampler()
updater_PF                = ParticleUpdater(measurement_model=tracking_measurement_model, resampler=resampler)
hypothesiser_PF           = DistanceHypothesiser(predictor_PF, updater_PF, measure=Mahalanobis(), missed_distance=5)
data_associator_PF        = GNNWith2DAssignment(hypothesiser_PF)
deleter                   = UpdateTimeDeleter(datetime.timedelta(seconds=1), delete_last_pred=True)
prior_state               = GaussianState(StateVector([0, 0, 0, 0, 0]), np.diag([50, 50, np.pi, 10, np.pi]) ** 2)
initiator_Part            = SimpleMeasurementInitiator(prior_state, measurement_model=tracking_measurement_model,skip_non_reversible=True)
initiator_PF              = GaussianParticleInitiator(number_particles=1000,initiator=initiator_Part, use_fixed_covar=False)
tracker_PF                = MultiTargetTracker(initiator=initiator_PF, deleter=deleter,detector=detector,data_associator=data_associator_PF,updater=updater_PF)

tracks_PF = set()
timesteps = []
for time, current_tracks in tqdm(tracker_PF,total=number_steps):
    tracks_PF.update(current_tracks)
    timesteps.append(time)


In [ ]:
plotter = AnimatedPlotterly(timesteps=timesteps, tail_length=0.3)
plotter.plot_measurements(detections, [0, 2], measurement_model=gen_measurement_model)
plotter.plot_ground_truths(ground_truth, [0, 2])
plotter.plot_tracks(tracks_PF, [0, 1], label="PF", uncertainty=False, particle=True)
plotter.fig

In [ ]:
plotter = AnimatedPlotterly(timesteps, tail_length=0.3)
plotter.plot_ground_truths(ground_truth, [0, 2])
plotter.plot_track_headings(tracks_PF, [0, 1, 2, 3], plot_history=True, velocity_scale=1)
plotter.fig

In [ ]:

transition_model_estimate = CombinedLinearGaussianTransitionModel([ConstantVelocity(1),ConstantVelocity(1)])
tracking_measurement_model= LinearGaussian(ndim_state=4,mapping=(0, 2), noise_covar=np.array([[2, 0], [0, 2]]))
predictor_PF              = ParticlePredictor(transition_model_estimate)
resampler                 = ESSResampler()
updater_PF                = ParticleUpdater(measurement_model=tracking_measurement_model, resampler=resampler)
hypothesiser_PF           = DistanceHypothesiser(predictor_PF, updater_PF, measure=Mahalanobis(), missed_distance=5)
data_associator_PF        = GNNWith2DAssignment(hypothesiser_PF)
deleter                   = UpdateTimeDeleter(datetime.timedelta(seconds=1), delete_last_pred=True)
prior_state               = GaussianState(StateVector([0, 0, 0, 0]), np.diag([50, 10, 50, 10]) ** 2)
initiator_Part            = SimpleMeasurementInitiator(prior_state, measurement_model=tracking_measurement_model,skip_non_reversible=True)
initiator_PF              = GaussianParticleInitiator(number_particles=1000,initiator=initiator_Part, use_fixed_covar=False)
tracker_PF                = MultiTargetTracker(initiator=initiator_PF, deleter=deleter,detector=detector,data_associator=data_associator_PF,updater=updater_PF)

tracks_PF = set()
timesteps = []
for time, current_tracks in tqdm(tracker_PF,total=number_steps):
    tracks_PF.update(current_tracks)
    timesteps.append(time)


plotter = AnimatedPlotterly(timesteps=timesteps, tail_length=0.3)
plotter.plot_measurements(detections, [0, 2], measurement_model=gen_measurement_model)
plotter.plot_ground_truths(ground_truth, [0, 2])
plotter.plot_tracks(tracks_PF, [0, 2], label="PF", uncertainty=False, particle=True)
plotter.fig



In [ ]:

tracking_filters = ['PF']

# %%
from stonesoup.metricgenerator.ospametric import OSPAMetric

ospa_generators = [OSPAMetric(c=40, p=1,
                              generator_name=f'{tracking_filter} OSPA metrics',
                              tracks_key=f'tracks_{tracking_filter}',
                              truths_key='truths'
                             )
                   for tracking_filter in tracking_filters]

from stonesoup.metricgenerator.tracktotruthmetrics import SIAPMetrics
from stonesoup.measures import Euclidean

siap_generators = [SIAPMetrics(position_measure=Euclidean((0, 2)),
                             velocity_measure=Euclidean((1, 3)),
                             generator_name=f'{tracking_filter} SIAP metrics',
                             tracks_key=f'tracks_{tracking_filter}',
                             truths_key='truths'
                            )
                  for tracking_filter in tracking_filters]


from stonesoup.metricgenerator.uncertaintymetric import SumofCovarianceNormsMetric

uncertainty_generators = [
    SumofCovarianceNormsMetric(generator_name=f'{tracking_filter} OSPA metrics',
                               tracks_key=f'tracks_{tracking_filter}')
    for tracking_filter in tracking_filters]

# %%
# Now we initialise the metric manager and generate the metrics:

from stonesoup.dataassociator.tracktotrack import TrackToTruth
from stonesoup.metricgenerator.manager import MultiManager

associator = TrackToTruth(association_threshold=30)

generators = ospa_generators + siap_generators + uncertainty_generators
metric_manager = MultiManager(generators, associator=associator)

metric_manager.add_data({'truths': ground_truth,
                         #'tracks_EKF': tracks_EKF,
                         #'tracks_UKF': tracks_UKF,
                         'tracks_PF': tracks_PF,
                         #'tracks_ESIF': tracks_ESIF
                         })
metrics = metric_manager.generate_metrics()

# %%
# Now we can plot the OSPA distance for each tracker:

from stonesoup.plotter import MetricPlotter

fig1 = MetricPlotter()
fig1.plot_metrics(metrics, metric_names=['OSPA distances'])

# %%
# It can be seen that the EKF, UKF, and Particle Filter all behave very similarly,
# whereas the ESIF has very poor relative performance. A singular performance metric is calculated
# from this by summing the OSPA value over all timesteps:

# sum up distance error from ground truth over all timestamps
for tracking_filter in tracking_filters:
    total = sum([metrics[f'{tracking_filter} OSPA metrics']['OSPA distances'].value[i].value
                 for i in range(0, len(metrics[f'{tracking_filter} OSPA metrics']['OSPA distances'].value))])
    print(f'OSPA total value for {tracking_filter} is {total:.3f}')

# %%
# Finally, we calculate the SIAP metrics for the EKF. The same metrics can be calculated for the
# other trackers if desired. The user can copy this section of code and replace the relevant
# variable names to get the full metrics for the UKF, ESIF, and Particle Filter.
from stonesoup.metricgenerator.metrictables import SIAPTableGenerator

# generate metrics for EKF
siap_metrics = metrics['EKF SIAP metrics']
siap_averages_EKF = {siap_metrics.get(metric) for metric in siap_metrics
                     if metric.startswith("SIAP") and not metric.endswith(" at times")}

_ = SIAPTableGenerator(siap_averages_EKF).compute_metric()
print("\n\nSIAP metrics for EKF:")
